In [1]:
import ast
import concurrent.futures
import glob
import itertools
import os
import pickle
import warnings
import sys

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator


import numpy as np
import pandas as pd
import polars as pl
import scipy as sp
import statsmodels.api as sm

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)


warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Read Data

In [2]:
scaled_numeric_even_combined_df = pl.read_parquet("scaled_numeric_even_combined_df.parquet").sort(["cutoff","fips"])
scaled_numeric_odd_combined_df = pl.read_parquet("scaled_numeric_odd_combined_df.parquet").sort(["cutoff","fips"])

In [3]:
even_indices = scaled_numeric_even_combined_df.select(["cutoff"]).unique().sort(["cutoff"])
odd_indices = scaled_numeric_odd_combined_df.select(["cutoff"]).unique().sort(["cutoff"])
combined_data = pl.concat([scaled_numeric_even_combined_df, scaled_numeric_odd_combined_df]).sort(["cutoff","fips"])

fips_list = scaled_numeric_odd_combined_df.select(["fips"]).unique().sort(["fips"])

all_indices = pl.concat([scaled_numeric_even_combined_df.select(["cutoff","fips"]).unique().sort(["cutoff","fips"]),scaled_numeric_odd_combined_df.select(["cutoff","fips"]).unique().sort(["cutoff","fips"])]).sort(["cutoff","fips"])

In [4]:
# Gram matrices
gram_matrices_subfolder = "./gram_matrices"
beta_subfolder = "./betas"

In [5]:
all_indices[0]

cutoff,fips
i64,i64
51,6085


In [6]:
# Columns to exclude
index_cols = ["fips", "cutoff"]
exclude_cols = ["State_FIPS_Code", "log_rolled_cases.x", "log_rolled_cases.y", "t0.lm", "r.lm"]
outcome_col = "shifted_log_rolled_cases"
feature_cols = list(set(scaled_numeric_even_combined_df.columns) - set(index_cols) - set(exclude_cols) - set([outcome_col]))


In [92]:
i = 764000
print(all_indices[i])

shape: (1, 2)
┌────────┬───────┐
│ cutoff ┆ fips  │
│ ---    ┆ ---   │
│ i64    ┆ i64   │
╞════════╪═══════╡
│ 410    ┆ 47019 │
└────────┴───────┘


In [93]:
if i % 2 == 0:
    dataset = scaled_numeric_even_combined_df
else:
    dataset = scaled_numeric_odd_combined_df

beta = np.load("./betas/beta_cutoff={}.npy".format(all_indices[i,0])).reshape(-1)
XtX = np.load("./gram_matrices/gram_matrix_cutoff={}.npy".format(all_indices[i,0]))

X = dataset.filter(pl.col("cutoff")  == all_indices[i,0]).select(feature_cols).with_columns(pl.lit(1).alias("intercept")).to_pandas().values
y = dataset.filter(pl.col("cutoff")  == all_indices[i,0]).select(outcome_col).to_pandas().values.reshape(-1)
Xty = np.dot(X.T,y)

V = dataset.filter((pl.col("cutoff")  == all_indices[i,0]) &(pl.col("fips")  == all_indices[i,1])).select(feature_cols).with_columns(pl.lit(1).alias("intercept")).to_pandas().values
U = V.T

X_exclude_i = dataset.filter((pl.col("cutoff")  == all_indices[i,0])&(pl.col("fips")  != all_indices[i,1])).select(feature_cols).with_columns(pl.lit(1).alias("intercept")).to_pandas().values
y_exclude_i = dataset.filter((pl.col("cutoff")  == all_indices[i,0]) &(pl.col("fips")  != all_indices[i,1])).select(outcome_col).to_pandas().values.reshape(-1)

xi = dataset.filter((pl.col("cutoff")  == all_indices[i,0]) &(pl.col("fips")  == all_indices[i,1])).select(feature_cols).with_columns(pl.lit(1).alias("intercept")).to_pandas().values
yi = dataset.filter((pl.col("cutoff")  == all_indices[i,0]) &(pl.col("fips")  == all_indices[i,1])).select(outcome_col).to_pandas().values.reshape(-1)
xtyi = np.dot(xi.T,yi)

In [94]:
beta.shape, XtX.shape, X.shape, xi.shape

((471,), (471, 471), (5220, 471), (2, 471))

In [95]:
# A = X'X
# U = V'
# V = X[[i_day0, i_day1],:]

# Block structure so rank2 update



def woodbury_matrix(A, U, V):
    I = np.eye(len(V))
    AU = A @ U
    return sp.linalg.blas.dgemm(
        alpha=-1,
        a=AU @ np.linalg.inv(I + V @ AU),
        b=V @ A,
        beta=1, c=A
    )

# Apply sm update twice
# B = inv(X'X)
def sm_fastest(B,u,v):
    # Warning: `overwrite_a=True` silently fails when B is not an order=F array!
    assert B.flags['F_CONTIGUOUS']
    Bu = B @ u
    alpha = -1 / (1 + v.T @ Bu)
    sp.linalg.blas.dger(alpha, Bu, v.T @ B, a=B, overwrite_a=1)
    return B


In [196]:
# Apply sm update twice

B = np.asfortranarray(np.linalg.inv(XtX + 1e4*np.eye(len(XtX))))
sm_fastest(B,-U[:,0],V[0,:])
#beta_i = np.dot(B, Xty - U[:,0]*yi[0] )
sm_fastest(XtX_i,-U[:,1],V[1,:])
beta_i = np.dot(B, np.dot(X_exclude_i.T,y_exclude_i))

In [197]:
np.dot(X_exclude_i.T,y_exclude_i).shape

(471,)

In [198]:
B.shape, U[:,0].shape, V[0,].shape

((471, 471), (471,), (471,))

In [199]:
X_exclude_i.shape, y_exclude_i.shape

((5218, 471), (5218,))

In [200]:
XtX

array([[ 1.91487500e+05, -1.94289029e-16,  4.98732999e-18, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-1.94289029e-16,  6.46600999e+04,  2.03633509e+03, ...,
         2.98389119e+01,  3.58634839e+01, -4.89551987e+03],
       [ 4.98732999e-18,  2.03633509e+03,  4.12085599e+03, ...,
         1.60791323e+01,  1.93255607e+01, -2.63802219e+03],
       ...,
       [ 0.00000000e+00,  2.98389119e+01,  1.60791323e+01, ...,
         2.84556657e+01,  3.42009559e+01, -4.66857763e+03],
       [ 0.00000000e+00,  3.58634839e+01,  1.93255607e+01, ...,
         3.42009559e+01,  4.11062386e+01, -5.61117844e+03],
       [ 0.00000000e+00, -4.89551987e+03, -2.63802219e+03, ...,
        -4.66857763e+03, -5.61117844e+03,  7.65950000e+05]])

In [201]:
B @ XtX

array([[ 9.50369145e-01,  4.45955417e-08, -1.07776384e-08, ...,
         6.11427109e-10,  7.34876205e-10, -1.00313764e-07],
       [ 2.84012176e-06,  3.71641037e-01,  7.74996627e-03, ...,
        -3.97963616e-05, -4.78313746e-05,  6.52918844e-03],
       [-3.56321347e-06,  7.74137721e-03,  4.80268076e-02, ...,
        -2.84196482e-05, -3.41576664e-05,  4.66266843e-03],
       ...,
       [-1.66566934e-07, -4.04099222e-05, -2.81536054e-05, ...,
         9.39229469e-06,  1.12886291e-05, -1.54094645e-03],
       [-9.86284554e-07, -5.09157600e-05, -3.32707103e-05, ...,
         1.12564513e-05,  1.35291650e-05, -1.84678923e-03],
       [-3.43185116e-06,  6.53801614e-03,  4.64121465e-03, ...,
        -1.54220557e-03, -1.85358182e-03,  2.53021894e-01]])

In [202]:
#https://stats.stackexchange.com/questions/449492/least-squares-removing-first-k-observations-woodbury-formula

In [203]:
I = np.eye(len(V))
I
#np.linalg.solve(H,res[i::k])

array([[1., 0.],
       [0., 1.]])

In [204]:
np.linalg.norm(beta)

1.109152935292417

In [205]:
np.linalg.norm(np.dot(np.linalg.inv(XtX+ 0.1*np.eye(len(XtX))),Xty))

230.91860399193826

In [206]:
beta_ridged = np.linalg.solve(XtX+ 1e4*np.eye(len(XtX)),Xty)
beta_ridged[-1]

-0.0011785223638163288

In [207]:
np.linalg.norm(beta_ridged)

0.003507281153360063

In [208]:
np.linalg.norm(beta_i)

0.0035023662511745822

In [209]:
y_pred_beta = np.dot(X,beta)

In [210]:
y

array([-0.0001493 , -0.00511684, -0.00698871, ..., -0.0001493 ,
       -0.0001493 , -0.00210709])

In [214]:
y_pred_beta_sm = np.dot(X,beta_i)

In [212]:
y_pred_beta_ridged = np.dot(X,beta_ridged)

In [216]:
mean_squared_error(y_pred_beta, y)**2

0.0007994226124492303

In [217]:
mean_squared_error(y_pred_beta_ridged, y)**2

2.7851636789799297e-09

In [218]:
mean_squared_error(y_pred_beta_sm, y)**2

2.7790306105480383e-09